# 4.1 Systematic Model Comparison: Logistic Regression vs. Random Forest vs. XGBoost

## Course 3: Advanced Classification Models for Student Success

## Introduction

Throughout this course, we have built three families of classification models:

1. **Module 2**: Regularized Logistic Regression (L1, L2, ElasticNet)
2. **Module 3**: Tree-Based Models (Decision Tree, Random Forest, XGBoost)

In this module, we bring the **three most practical models** together for a head-to-head comparison:

- **Regularized Logistic Regression** — the interpretable baseline
- **Random Forest** — the robust ensemble
- **XGBoost** — the performance champion

### Why These Three?

These represent the models you will use most often in practice. They cover the full spectrum of the interpretability-performance trade-off, and each excels in different scenarios. Other models (LightGBM, CatBoost, Neural Networks) are covered in the Special Topics module.

### Learning Objectives

1. Train and evaluate all three models on the same dataset with optimized hyperparameters
2. Compare performance across multiple metrics (AUC, F1, Precision, Recall)
3. Analyze the interpretability vs. performance trade-off
4. Apply model selection criteria appropriate for higher education
5. Make a justified recommendation for deployment

## 1. Setup and Data Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --- Paths ---
project_path  = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3'
data_filepath = '/data/'
course3_models = '/models/'

In [ ]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, brier_score_loss, log_loss, classification_report)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Pull feature columns and training median artifacts
ARTIFACT_DIR = f'{project_path}{course3_models}'
feature_columns = joblib.load(f'{ARTIFACT_DIR}feature_columns.pkl')
train_medians   = joblib.load(f'{ARTIFACT_DIR}train_medians.pkl')

# --- Load TEST data only ---
test_df = pd.read_csv(f'{project_path}{data_filepath}testing.csv')
test_df['DEPARTED'] = (test_df['SEM_3_STATUS'] != 'E').astype(int)

numeric_features = ['HS_GPA','HS_MATH_GPA','HS_ENGL_GPA','UNITS_ATTEMPTED_1','UNITS_ATTEMPTED_2',
    'UNITS_COMPLETED_1','UNITS_COMPLETED_2','DFW_UNITS_1','DFW_UNITS_2','GPA_1','GPA_2',
    'DFW_RATE_1','DFW_RATE_2','GRADE_POINTS_1','GRADE_POINTS_2']
categorical_features = ['RACE_ETHNICITY','GENDER','FIRST_GEN_STATUS','COLLEGE']

# --- Raw features for the logistic PIPELINE (it encodes + scales internally) ---
X_test_raw = test_df[numeric_features + categorical_features].copy()

# --- Encoded matrix for the TREE models (manual dummies, aligned to training columns) ---
test_enc = pd.get_dummies(test_df[numeric_features + categorical_features],
                          columns=categorical_features, drop_first=True)
test_enc = test_enc.reindex(columns=feature_columns, fill_value=0)  # match training feature set
test_enc = test_enc.fillna(train_medians)                          # impute with TRAIN medians

X_test, y_test = test_enc, test_df['DEPARTED']

y_test = test_df['DEPARTED']

print(f"Testing: {X_test.shape[0]:,} | Encoded features: {X_test.shape[1]} | Raw cols: {X_test_raw.shape[1]}")
print(f"Departure rate: {y_test.mean():.2%} (test)")

Testing: 5,336 | Encoded features: 31 | Raw cols: 19
Departure rate: 15.37% (test)


## 2. Load the Regularized Regression and Tree Based Models

In [ ]:
import joblib

# --- Load models ---
lr = joblib.load(f'{ARTIFACT_DIR}best_tuned_logistic_model.pkl')
rf = joblib.load(f'{ARTIFACT_DIR}rf_tuned_f1.pkl')
xgb = joblib.load(f'{ARTIFACT_DIR}xgb_tuned_f1.pkl')

# --- Generate predictions and probabilities ---

# Logistic Regression
lr_pred = (lr.predict(X_test_raw) != 'E').astype(int)
e_class_idx = list(lr.classes_).index('E')
lr_prob = lr.predict_proba(X_test_raw)[:, 1]

# Random Forest
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

# XGBoost
xgb_pred = xgb.predict(X_test)
xgb_prob = xgb.predict_proba(X_test)[:, 1]

print("Successfully loaded models and generated predictions/probabilities.")
print(f"Unique values for Logistic Regression predictions: {np.unique(lr_pred)}")
print(f"Unique values for Random Forest predictions: {np.unique(rf_pred)}")
print(f"Unique values for XGBoost predictions: {np.unique(xgb_pred)}")

Successfully loaded models and generated predictions/probabilities.
Unique values for Logistic Regression predictions: [0 1]
Unique values for Random Forest predictions: [0 1]
Unique values for XGBoost predictions: [0 1]


In [ ]:
X_test_raw.head()

,HS_GPA,HS_MATH_GPA,HS_ENGL_GPA,UNITS_ATTEMPTED_1,UNITS_ATTEMPTED_2,UNITS_COMPLETED_1,UNITS_COMPLETED_2,DFW_UNITS_1,DFW_UNITS_2,GPA_1,GPA_2,DFW_RATE_1,DFW_RATE_2,GRADE_POINTS_1,GRADE_POINTS_2,RACE_ETHNICITY,GENDER,FIRST_GEN_STATUS,COLLEGE
0,3.906,3.833,3.800,15.0,14.0,15.0,15.0,0.0,0.0,3.333333,3.642857,0.0000,0.000000,50.0,51.0,Other,Female,Continuing Generation,Natural and Mathematical Sciences
1,3.531,3.500,3.667,15.0,14.0,15.0,11.0,0.0,3.0,3.600000,2.357143,0.0000,0.214286,54.0,33.0,Hispanic,Female,First Generation,Education & Leadership
2,3.543,3.667,3.500,14.0,15.0,15.0,15.0,0.0,0.0,2.071429,3.600000,0.0000,0.000000,29.0,54.0,Hispanic,Male,First Generation,Letters & Humanities
3,3.417,3.200,2.750,10.0,13.0,9.0,13.0,1.0,0.0,3.000000,2.461538,0.1000,0.000000,30.0,32.0,Hispanic,Male,Continuing Generation,Natural and Mathematical Sciences
4,3.583,3.200,3.800,16.0,16.0,9.0,0.0,7.0,16.0,1.937500,0.625000,0.4375,1.000000,31.0,10.0,Hispanic,Male,First Generation,Letters & Humanities


## 3. Performance Comparison

In [ ]:
model_results = []

# Define the models, their predictions, and probabilities
models = {
    'Regularized Logistic': {'preds': lr_pred, 'probs': lr_prob},
    'Random Forest': {'preds': rf_pred, 'probs': rf_prob},
    'XGBoost': {'preds': xgb_pred, 'probs': xgb_prob}
}

for name, data in models.items():
    preds = data['preds']
    probs = data['probs']

    # Calculate metrics
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    avg_precision = average_precision_score(y_test, probs)
    brier = brier_score_loss(y_test, probs)

    model_results.append({
        'Model': name,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'Avg Precision': avg_precision,
        'Brier Score': brier
    })

# Create DataFrame
results_df = pd.DataFrame(model_results).set_index('Model')

# Print formatted output
header = "HEAD-TO-HEAD MODEL COMPARISON"
separator = "=" * 90

print(separator)
print(f"{header:^90}")
print(separator)
print(results_df.to_string(float_format="%.4f"))
print(separator)

                              HEAD-TO-HEAD MODEL COMPARISON                               
                      Precision  Recall  F1 Score  Avg Precision  Brier Score
Model                                                                        
Regularized Logistic     0.4673  0.7049    0.5620         0.6395       0.1488
Random Forest            0.8000  0.6439    0.7135         0.7655       0.0715
XGBoost                  0.6923  0.6915    0.6919         0.7624       0.0813


## 4. Precision-Recall Curves

In [ ]:
import plotly.graph_objects as go
from sklearn.metrics import precision_recall_curve, average_precision_score

# Define a colors list of three distinct hex colors
colors = ['#1f77b4', '#ff7f0e', '#2ca02c'] # Blue, Orange, Green

fig = go.Figure()

for i, (name, prob) in enumerate([('Regularized Logistic', lr_prob),
                                    ('Random Forest', rf_prob), ('XGBoost', xgb_prob)]):
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    fig.add_trace(go.Scatter(x=rec, y=prec, mode='lines',
        name=f'{name} (AP={ap:.3f})', line=dict(color=colors[i], width=2)))

# No-skill baseline = positive-class prevalence (departure rate)
baseline = y_test.mean()
fig.add_trace(go.Scatter(x=[0, 1], y=[baseline, baseline], mode='lines',
    name=f'No-skill ({baseline:.3f})', line=dict(color='gray', dash='dash')))

fig.update_layout(height=450, title_text='Precision-Recall Curves')
fig.update_xaxes(title_text='Recall')
fig.update_yaxes(title_text='Precision')
fig.show()

## 5. Interpretability vs. Performance Trade-off

In [8]:
import plotly.graph_objects as go
import pandas as pd

# --- 1. Get F1 scores and scale to 0-10 ---
max_f1 = results_df['F1 Score'].max()
f1_lr = (results_df.loc['Regularized Logistic', 'F1 Score'] / max_f1) * 10
f1_rf = (results_df.loc['Random Forest', 'F1 Score'] / max_f1) * 10
f1_xgb = (results_df.loc['XGBoost', 'F1 Score'] / max_f1) * 10

# --- 2. Define subjective scores (0-10 scale) ---
dimensions = [
    'F1 (scaled to 10)',
    'Interpretability',
    'Training Speed',
    'Handles Non-linearity',
    'Ease of Deployment'
]

data = {
    'Regularized Logistic': [
        f1_lr,  # F1
        9,      # Interpretability (high)
        9,      # Training Speed (fast)
        3,      # Handles Non-linearity (low)
        9       # Ease of Deployment (easy)
    ],
    'Random Forest': [
        f1_rf,  # F1
        5,      # Interpretability (medium)
        7,      # Training Speed (medium)
        8,      # Handles Non-linearity (high)
        7       # Ease of Deployment (medium)
    ],
    'XGBoost': [
        f1_xgb, # F1
        3,      # Interpretability (low)
        6,      # Training Speed (medium-slow)
        9,      # Handles Non-linearity (very high)
        6       # Ease of Deployment (medium-hard)
    ]
}

df_radar = pd.DataFrame(data, index=dimensions)

# --- 3. Create Plotly Radar Chart ---
fig = go.Figure()

for col in df_radar.columns:
    fig.add_trace(go.Scatterpolar(
        r=df_radar[col].values,
        theta=df_radar.index,
        fill='toself',
        name=col,
        opacity=0.3,
        line_width=2
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 10]
        )
    ),
    showlegend=True,
    title_text='Model Comparison: Performance vs. Practicality'
)

fig.show()

## 6. Recommendations for Higher Education

### Decision Framework

| Your Priority | Recommended Model | Why |
|:-------------|:-----------------|:----|
| **Explainability to advisors** | Regularized Logistic Regression | Coefficients show clear factor contributions |
| **Reliable risk scoring** | Random Forest | Robust, handles messy data well |
| **Maximum predictive accuracy** | XGBoost | Typically highest AUC and F1 |
| **Research publications** | XGBoost | Best metrics for academic papers |

### Practical Recommendation

For most higher education institutions, we recommend a **two-model approach**:

1. **Regularized Logistic Regression** for stakeholder-facing outputs (reports, advisor dashboards, compliance)
2. **Random Forest or XGBoost** for backend risk scoring where performance matters most

This gives you the best of both worlds: interpretability where it's needed, and performance where it counts.

## 7. Summary

### Key Findings

1. All three models are strong performers on student departure prediction
2. The performance gap between models is often smaller than expected—interpretability and deployment considerations often matter more
3. Regularized Logistic Regression remains highly competitive while being fully transparent
4. XGBoost typically edges ahead on raw metrics but requires more infrastructure
5. Random Forest provides an excellent middle ground

### Next Module

**Proceed to:** `Module 5: Unsupervised Learning`